In [2]:
using Plots
using Measures
using JuMP, GLPK, CPLEX

# Exercise 1
hospitals surgery plan for $n$ days
$S$...Set of Surgeries
$D={1,...n}$...Set of days
For $s \in S$:
- $t_s$...**hours** of operating-room time
- $r_s$...prefered **latest completion day**
- $p_s$...**penalty** cost for each day, the surgery is late
For $d \in D$:
- $H_d$...**capacity** of operating-room hours 

every surgery must be scheduled exactly once during the $n$ days



# Question 1
formulate the Mixed-Integer Linear Programing model, that minimizes the total delay penalty

In [4]:
# EXERCISE 1 - Modelling
# Question 1: Hospital hours

model = Model(GLPK.Optimizer) 

S = [1,2,3,4,5,6,7,8,9]    
M = length(S)   #amount of surgeries
t = [7,6,3,5,1,4,3,1,1]
r = [1,1,2,3,4,1,1,7,5]
p = [5,2,2,2,2,1,1,1,1]

N = 7       #amount of days
D = [i for i=1:N]
H = [9,7,2,5,7,8,9]

@variable(model,x[1:M,1:N], Bin)

@objective(model, Min, sum( sum( p[s] * x[s,d] * max(d - r[s],0) for s=1:M) for d=1:N) )
# Defining the constraints
@constraint(model, [s=1:M], sum( x[s,d] for d=1:N) == 1)
@constraint(model, [d=1:N], sum( x[s,d] * t[s] for s=1:M) <= H[d])

optimize!(model)                   

## Printing the solution
println("Objective value = ", objective_value(model)) 

println("Solver status: ", termination_status(model))
println("Solution status: ", primal_status(model))

print("feasible solution:")
for s=1:M
    for d=1:N
        print(value(x[s,d]))
        print("  ")
    end
    print("\n")
end
print("- each row stands for a surgery\n- each column stands for a day")

Objective value = 15.0
Solver status: OPTIMAL
Solution status: FEASIBLE_POINT
feasible solution:0.0  1.0  0.0  0.0  0.0  0.0  0.0  
1.0  0.0  0.0  0.0  0.0  0.0  0.0  
1.0  0.0  0.0  0.0  0.0  0.0  0.0  
0.0  0.0  0.0  1.0  0.0  0.0  0.0  
0.0  0.0  1.0  0.0  0.0  0.0  0.0  
0.0  0.0  0.0  0.0  1.0  0.0  0.0  
0.0  0.0  0.0  0.0  1.0  0.0  0.0  
0.0  0.0  0.0  0.0  0.0  1.0  0.0  
0.0  0.0  1.0  0.0  0.0  0.0  0.0  
- each row stands for a surgery
- each column stands for a day

# Question 2
additional Constraints:
- $K$...Set of surgeons
- $S_k \cup S$...Set of surgeries that require surgeon $k$
- A surgeon must not work three days in a row

Formulate the new Mixed-Integer Linear Programing model, that minimizes the total delay penalty and considers the additional constraints

In [ ]:
# Question 2: Surgeons workdays 
using Plots
using Measures
using JuMP, GLPK, CPLEX
model = Model(GLPK.Optimizer) 

S = [1,2,3,4,5,6,7,8,9]    
M = length(S)   #amount of surgeries
t = [7,6,3,5,1,4,3,1,1]
r = [1,1,2,3,4,1,1,7,5]
p = [5,2,2,2,2,1,1,1,1]

K = 3
Sk = [[1,2,4,5,7,8], # S1
    [3,6],           # S2
    [9]]             # S3

N = 7       #amount of days
D = [i for i=1:N]
H = [9,7,2,5,7,8,9]

@variable(model,x[1:M,1:N], Bin)
@variable(model,y[1:3,1:N], Bin)

@objective(model, Min, sum( sum( p[s] * x[s,d] * max(d - r[s],0) for s=1:M) for d=1:N) )
# Defining the constraints
@constraint(model, [s=1:M], sum( x[s,d] for d=1:N) == 1)
@constraint(model, [d=1:N], sum( x[s,d] * t[s] for s=1:M) <= H[d])

#surgeon
@constraint(model,[d=1:N-2,k=1:3], y[k,d] + y[k,d+1] + y[k,d+2] <= 2)

for k = 1:K
    for d = 1:N
        for s in Sk[k]
            @constraint(model, x[s,d] <= y[k,d])
        end
    end
end
#Solving the model
optimize!(model)                   

## Printing the solution
println("Objective value = ", objective_value(model)) 

println("Solver status: ", termination_status(model))
println("Solution status: ", primal_status(model))

println("SURGERY SCHEDULE:")
for s=1:M
    for d=1:N
        print(value(x[s,d]))
        print("  ")
    end
    print("\n")
end

println("-> each row stands for a surgery\n-> each column stands for a day\n")

println("SURGEONS SCHEDULE:")
for k=1:3
    for d=1:N
        print(value(y[k,d]))
        print("  ")
    end
    print("\n")
end
print("-> each row stands for a surgeon\n-> each column stands for a day")


Objective value = 16.0
Solver status: OPTIMAL
Solution status: FEASIBLE_POINT
SURGERY SCHEDULE:
1.0  0.0  0.0  0.0  0.0  0.0  0.0  
0.0  0.0  0.0  0.0  1.0  0.0  0.0  
0.0  1.0  0.0  0.0  0.0  0.0  0.0  
0.0  0.0  0.0  1.0  0.0  0.0  0.0  
0.0  1.0  0.0  0.0  0.0  0.0  0.0  
0.0  0.0  0.0  0.0  0.0  1.0  0.0  
0.0  1.0  0.0  0.0  0.0  0.0  0.0  
0.0  0.0  0.0  0.0  0.0  0.0  1.0  
0.0  0.0  0.0  0.0  1.0  0.0  0.0  
-> each row stands for a surgery
-> each column stands for a day

SURGEONS SCHEDULE:
1.0  1.0  0.0  1.0  1.0  0.0  1.0  
0.0  1.0  0.0  0.0  1.0  1.0  0.0  
0.0  0.0  1.0  0.0  1.0  0.0  0.0  
-> each row stands for a surgeon
-> each column stands for a day

## Question 2 *-> ALTERNATIVE*

In [15]:
# Question 2: Surgeons workdays - ALTERNATIVE: worked with a list K instead of sets S_k
model = Model(GLPK.Optimizer) 

S = [1,2,3,4,5,6,7,8,9]    
M = length(S)   #amount of surgeries
t = [7,6,3,5,1,4,3,1,1]
r = [1,1,2,3,4,1,1,7,5]
p = [5,2,2,2,2,1,1,1,1]
# 3 surgeons, each surgery needs one of them:
K = [1,1,2,1,1,2,1,1,3]

N = 7           #amount of days
D = [i for i=1:N]
H = [9,7,2,5,7,8,9]

@variable(model,x[1:M,1:N], Bin)
@variable(model,y[1:3,1:N], Bin)

@objective(model, Min, sum( sum( p[s] * x[s,d] * max(d - r[s],0) for s=1:M) for d=1:N) )
# Defining the constraints
@constraint(model, [s=1:M], sum( x[s,d] for d=1:N) == 1)
@constraint(model, [d=1:N], sum( x[s,d] * t[s] for s=1:M) <= H[d])

#surgeon
@constraint(model,[d=1:N-2,k=1:3], y[k,d] + y[k,d+1] + y[k,d+2] <= 2)
@constraint(model, [d=1:N,s=1:M], x[s,d] <= y[ K[s] , d ])

optimize!(model)                   

## Printing the solution
println("Objective value = ", objective_value(model)) 

println("Solver status: ", termination_status(model))
println("Solution status: ", primal_status(model))

println("SURGERY SCHEDULE:")
for s=1:M
    for d=1:N
        print(value(x[s,d]))
        print("  ")
    end
    print("\n")
end

println("-> each row stands for a surgery\n-> each column stands for a day\n")

println("SURGEONS SCHEDULE:")
for k=1:3
    for d=1:N
        print(value(y[k,d]))
        print("  ")
    end
    print("\n")
end
print("-> each row stands for a surgeon\n-> each column stands for a day")


Objective value = 16.0
Solver status: OPTIMAL
Solution status: FEASIBLE_POINT
SURGERY SCHEDULE:
1.0  0.0  0.0  0.0  0.0  0.0  0.0  
0.0  0.0  0.0  0.0  1.0  0.0  0.0  
0.0  1.0  0.0  0.0  0.0  0.0  0.0  
0.0  0.0  0.0  1.0  0.0  0.0  0.0  
0.0  1.0  0.0  0.0  0.0  0.0  0.0  
0.0  0.0  0.0  0.0  0.0  1.0  0.0  
0.0  1.0  0.0  0.0  0.0  0.0  0.0  
0.0  0.0  0.0  0.0  0.0  0.0  1.0  
0.0  0.0  0.0  0.0  1.0  0.0  0.0  
-> each row stands for a surgery
-> each column stands for a day

SURGEONS SCHEDULE:
1.0  1.0  0.0  1.0  1.0  0.0  1.0  
0.0  1.0  0.0  0.0  1.0  1.0  0.0  
0.0  0.0  1.0  0.0  1.0  0.0  0.0  
-> each row stands for a surgeon
-> each column stands for a day

# Question 3 
*...reserve capacity for emergencies*

$\roh$ hours of operating-room capacity must remain unused on at least $80%$ of the days

In [ ]:
# Question 3: Emergency Capacity
using Plots
using Measures
using JuMP, GLPK, CPLEX
model = Model(GLPK.Optimizer) 

#emergency hour
roh = 1

S = [1,2,3,4,5,6,7,8,9]    
M = length(S)   #amount of surgeries
t = [7,6,3,5,1,4,3,1,1]
r = [1,1,2,3,4,1,1,7,5]
p = [5,2,2,2,2,1,1,1,1]


N = 7       #amount of days
D = [i for i=1:N]
H = [9,7,2,5,7,8,9]

@variable(model,x[1:M,1:N], Bin)
@variable(model,c[1:N],Bin)

@objective(model, Min, sum( sum( p[s] * x[s,d] * max(d - r[s],0) for s=1:M) for d=1:N) )
# Defining the constraints
@constraint(model, [s=1:M], sum( x[s,d] for d=1:N) == 1)
# @constraint(model, [d=1:N], sum( x[s,d] * t[s] for s=1:M) <= H[d])

# emergency capacity
@constraint(model,[d=1:N], sum( x[s,d] * t[s] for s=1:M) <= H[d] - (c[d]*roh))
@constraint(model, sum( c[d] for d=1:N)/N >= 0.8)
#Solving the model
optimize!(model)                   

## Printing the solution
println("Objective value = ", objective_value(model)) 

println("Solver status: ", termination_status(model))
println("Solution status: ", primal_status(model))

for s=1:M
    for d=1:N
        print(value(x[s,d]))
        print("  ")
    end
    print("\n")
end
println("emercency days:")
for d=1:N
    print(value(c[d]))
    print("  ")
end
print("\n")

Objective value = 16.0
Solver status: OPTIMAL
Solution status: FEASIBLE_POINT
1.0  0.0  0.0  0.0  0.0  0.0  0.0  
0.0  0.0  0.0  0.0  1.0  0.0  0.0  
0.0  1.0  0.0  0.0  0.0  0.0  0.0  
0.0  0.0  0.0  1.0  0.0  0.0  0.0  
1.0  0.0  0.0  0.0  0.0  0.0  0.0  
0.0  0.0  0.0  0.0  0.0  1.0  0.0  
0.0  1.0  0.0  0.0  0.0  0.0  0.0  
0.0  0.0  0.0  0.0  0.0  1.0  0.0  
0.0  0.0  1.0  0.0  0.0  0.0  0.0  
emercency days:
1.0  1.0  1.0  0.0  1.0  1.0  1.0  


In [33]:
# Question 4: Robotic system
using Plots
using Measures
using JuMP, GLPK, CPLEX
model = Model(GLPK.Optimizer) 

gamma = 3 #costs to activate robotic system

S = [1,2,3,4,5,6,7,8,9]  
R = [0,1,1,1,0,0,0,0,0]  # whether we need the robotic system for the specific surgeries or not
M = length(S)   #amount of surgeries
t = [7,6,3,5,1,4,3,1,1]
r = [1,1,2,3,4,1,1,7,5]
p = [5,2,2,2,2,1,1,1,1]

N = 7       #amount of days
D = [i for i=1:N]
H = [9,7,2,5,7,8,9]

@variable(model,x[1:M,1:N], Bin)
@variable(model,z[1:N],Bin)

@objective(model, Min, sum( sum( p[s] * x[s,d] * max(d - r[s],0) for s=1:M) + gamma * z[d] for d=1:N) )
# Defining the constraints
@constraint(model, [s=1:M], sum( x[s,d] for d=1:N) == 1)
@constraint(model, [d=1:N], sum( x[s,d] * t[s] for s=1:M) <= H[d])

# robot-constraint
@constraint(model, [s=1:M,d=1:N], x[s,d] * R[s] <= z[d])
#Solving the model
optimize!(model)                   

## Printing the solution
println("Objective value = ", objective_value(model)) 

println("Solver status: ", termination_status(model))
println("Solution status: ", primal_status(model))

for s=1:M
    for d=1:N
        print(value(x[s,d]))
        print("  ")
    end
    print("\n")
end

println("Robotic system - turn on/off:")
for d=1:N
    print(value(z[d]))
    print("  ")
end


Objective value = 21.0
Solver status: OPTIMAL
Solution status: FEASIBLE_POINT
0.0  1.0  0.0  0.0  0.0  0.0  0.0  
1.0  0.0  0.0  0.0  0.0  0.0  0.0  
1.0  0.0  0.0  0.0  0.0  0.0  0.0  
0.0  0.0  0.0  1.0  0.0  0.0  0.0  
0.0  0.0  1.0  0.0  0.0  0.0  0.0  
0.0  0.0  0.0  0.0  1.0  0.0  0.0  
0.0  0.0  0.0  0.0  1.0  0.0  0.0  
0.0  0.0  0.0  0.0  0.0  1.0  0.0  
0.0  0.0  1.0  0.0  0.0  0.0  0.0  
Robotic system - turn on/off:
1.0  0.0  0.0  1.0  0.0  0.0  0.0  